# Práctico 1 – Primera llamada a un LLM

**Magíster en Inteligencia Artificial UC · INF3880 – Inteligencia Artificial Agéntica**

**Profesores:** Vicente Domínguez, Antonio Ossa · **Ayudante:** Michelle Madrid

**Nombre del alumno:** ESCRIBIR NOMBRE AQUÍ

> Este práctico **no es evaluado**. Son ejercicios propuestos para trabajar en el Módulo 2 de la clase 1 (lunes 19/10/2026) y terminar en casa si hace falta. La versión con soluciones se publica después de la clase.

En este práctico dejarás listo tu entorno y harás tu primera llamada a un modelo de lenguaje desde Python, usando la **API gratuita de Gemini**: verás qué son los **tokens**, cuánto costaría una llamada y qué hace el parámetro `temperature`. Es el banco de trabajo que usaremos en todo el curso.


## Configuración inicial

Usaremos la **API de Gemini en su nivel gratuito** (no requiere tarjeta de crédito). Cada alumno crea su propia key; toma unos 2 minutos.

### Paso A · Crear tu API key en Google AI Studio
1. Abre **https://aistudio.google.com/apikey** e inicia sesión con tu cuenta de Google (personal; la cuenta institucional puede tener AI Studio bloqueado).
2. La primera vez te pedirá **aceptar los términos de servicio** de la Gemini API. Al aceptarlos, Google crea automáticamente un proyecto por defecto.
3. Haz clic en **"Create API key"** (si ya tienes un proyecto, elige "Create API key in existing project" o el proyecto por defecto).
4. Copia la key con el botón **"Copiar clave"** (las keys actuales empiezan con `AQ.`; las antiguas con `AIza`). Guárdala en un lugar seguro: la vas a necesitar en el paso B. Si la pierdes, puedes crear otra desde la misma página.
5. Opcional pero recomendado: en la lista de keys, pasa el mouse sobre la etiqueta **"Unrestricted"** → **"Add restrictions"** → **"Restrict to Gemini API only"** → **"Restrict key"**.

> ¿Dónde ver los límites y el uso? En AI Studio, menú lateral → **Dashboard** → **Usage & Billing** (o https://aistudio.google.com/usage). Ahí aparece el plan "Free" y las cuotas por modelo.
>
> ¿Problemas? Si el botón "Create API key" no aparece, suele ser porque la cuenta es de una organización (Workspace) que restringe AI Studio: usa una cuenta personal de Gmail. Documentación oficial: https://ai.google.dev/gemini-api/docs/api-key

### Paso B · Guardar la key en Secrets de Colab
1. En este notebook, haz clic en el ícono de la **llave 🔑** de la barra lateral izquierda ("Secrets").
2. **"+ Add new secret"**: en *Name* escribe exactamente `GEMINI_API_KEY`; en *Value* pega la key.
3. Activa el interruptor **"Notebook access"** para este notebook.
4. **Nunca pegues la key en una celda, ni la subas a un repositorio, ni la compartas.** Trátala como una contraseña: si se filtra, crea una nueva y borra la antigua en AI Studio.

### Paso C · Instalar y configurar
Instalamos la librería `google-genai` y definimos el modelo en una sola variable `MODEL` para poder cambiarlo sin tocar el resto del notebook.

> ⚠️ **Dos cosas que debes saber del nivel gratuito.** (1) Según los términos de Google, los prompts y respuestas del nivel sin pago **pueden usarse para mejorar sus productos y ser revisados por personas**: no envíes datos personales, confidenciales ni de tu empresa. (2) Tiene **límites de uso por minuto y por día**; si recibes un error 429 (*resource exhausted*), espera un momento y reintenta. Los límites de tu proyecto se ven en AI Studio.

In [ ]:
%%capture
!pip install -q --upgrade google-genai pandas

In [ ]:
import os, time
from google.colab import userdata

# 1) Leer la key desde Secrets, con mensajes claros si algo falta
try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except userdata.SecretNotFoundError:
    raise SystemExit("No existe un secreto llamado GEMINI_API_KEY. Créalo en el panel de Secrets (ícono de la llave) con ese nombre exacto.")
except userdata.NotebookAccessError:
    raise SystemExit("El secreto GEMINI_API_KEY existe pero no tiene 'Notebook access' activado para este notebook.")
assert GEMINI_API_KEY and len(GEMINI_API_KEY) > 20, \
    "El secreto GEMINI_API_KEY está vacío. Pega la key completa desde https://aistudio.google.com/apikey (botón 'Copiar clave')."
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# 2) Elegir el modelo: buscamos uno que esté disponible en TU proyecto gratuito y que respete `temperature`
#    (los modelos Gemini 3.6+ y 4 la ignoran, por eso preferimos Flash-Lite 3.5 o 2.5)
from google import genai
client = genai.Client()
PREFERIDOS = ["gemini-3.5-flash-lite", "gemini-2.5-flash-lite", "gemini-3.1-flash-lite", "gemini-2.5-flash", "gemini-3.8-flash"]
disponibles = []
for m in client.models.list():
    acciones = getattr(m, "supported_actions", None) or []
    if not acciones or "generateContent" in acciones:
        disponibles.append(m.name.replace("models/", ""))
MODEL = next((p for p in PREFERIDOS if p in disponibles), None)
if MODEL is None:
    print("Ninguno de los modelos preferidos está disponible. Modelos de tu proyecto:", disponibles[:30])
    MODEL = next((d for d in disponibles if "flash" in d), disponibles[0])
print("Modelo elegido:", MODEL)

# Precios de referencia del nivel DE PAGO, en USD por 1M tokens (https://ai.google.dev/gemini-api/docs/pricing).
# En el nivel gratuito no pagas, pero calcularemos cuánto costaría para entender la economía de un LLM.
PRECIOS = {
    "gemini-3.5-flash-lite": {"input": 0.30, "output": 2.50},
    "gemini-3.8-flash":      {"input": 0.75, "output": 3.75},
    "gemini-3.1-flash-lite": {"input": 0.25, "output": 1.50},
    "gemini-2.5-flash":      {"input": 0.30, "output": 2.50},
    "gemini-2.5-flash-lite": {"input": 0.10, "output": 0.40},
}

def costo_usd(input_tokens, output_tokens, model=None):
    p = PRECIOS.get(model or MODEL, PRECIOS["gemini-3.5-flash-lite"])   # si el modelo no está en la tabla, usa un precio de referencia
    return ((input_tokens or 0) * p["input"] + (output_tokens or 0) * p["output"]) / 1_000_000

print("Listo. Modelo:", MODEL)

Verificamos que la key funciona:

In [ ]:
from google.genai import types

def generar(contents, reintentos=2, **config):
    """Llama al modelo; si el nivel gratuito devuelve 429 (límite de uso), espera y reintenta."""
    for intento in range(reintentos + 1):
        try:
            return client.models.generate_content(model=MODEL, contents=contents,
                                                  config=types.GenerateContentConfig(**config))
        except Exception as e:
            msg = str(e)
            if ("429" in msg or "RESOURCE_EXHAUSTED" in msg) and intento < reintentos:
                print("  (límite de uso alcanzado, esperando 20 s...)"); time.sleep(20)
            elif "404" in msg or "NOT_FOUND" in msg:
                raise SystemExit(f"El modelo {MODEL} no está disponible en tu proyecto. Cambia MODEL por otro de la lista de la celda anterior.")
            elif "403" in msg or "401" in msg or "API_KEY_INVALID" in msg:
                raise SystemExit("La API key fue rechazada. Verifica que la copiaste completa desde https://aistudio.google.com/apikey y que no está restringida a otra API.")
            else:
                raise

print(generar("Responde solo: OK").text)

## Parte guiada 1 · Anatomía de una llamada

Una llamada a un LLM tiene tres partes: la **instrucción de sistema** (el rol y las reglas, lo que en la clase llamamos *system prompt*), el **contenido** que envía el usuario y los **parámetros** de generación.

In [ ]:
respuesta = generar(
    "¿Qué es un token?",
    system_instruction="Eres un tutor de un curso de IA agéntica. Respondes en español, en máximo 3 frases.",
    temperature=0.2,
    max_output_tokens=200,
)
print(respuesta.text)

💡 **Observa:** la respuesta viene en `.text`. Cambia `system_instruction` (por ejemplo, "responde como pirata") y vuelve a ejecutar: el mismo contenido produce una respuesta distinta. El *system prompt* es la especificación del comportamiento.

## Parte guiada 2 · Tokens y costo

El modelo no procesa palabras sino **tokens**, de entrada y de salida. El objeto `usage_metadata` trae el conteo exacto. En el nivel gratuito no pagas, pero calcularemos cuánto **costaría** en el nivel de pago para entender la economía de un LLM.

In [ ]:
u = respuesta.usage_metadata
print("tokens de entrada (prompt)   :", u.prompt_token_count)
print("tokens de salida (respuesta) :", u.candidates_token_count)
print("tokens de razonamiento       :", u.thoughts_token_count or 0, "  <- algunos modelos 'piensan' antes de responder y eso también son tokens")
print("total                        :", u.total_token_count)
salida = (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)
print(f"costaría (nivel de pago)     : US${costo_usd(u.prompt_token_count, salida):.6f}")

In [ ]:
# ¿Cuántos tokens ocupa un texto? Gemini permite contarlos sin generar nada (y sin consumir cuota de generación).
def contar_tokens(texto):
    return client.models.count_tokens(model=MODEL, contents=texto).total_tokens

for t in ["Hola", "Los agentes usan herramientas.", "Agents use tools.", "Supercalifragilístico"]:
    print(f"{t!r:40} -> {contar_tokens(t)} tokens")

💡 **Observa:** la misma frase en español suele ocupar más tokens que en inglés, y una palabra rara se parte en varios tokens. Google indica como regla aproximada ~4 caracteres por token. Esto afecta el costo y la ventana de contexto.

## Parte guiada 3 · `temperature`

`temperature` controla la aleatoriedad. Con 0 el modelo es casi determinista; con valores altos, más variado. Ejecutamos el mismo prompt varias veces.

> Nota: los modelos Gemini 3.6+ y 4 **ignoran** `temperature`; por eso este práctico usa `gemini-3.5-flash-lite`.

In [ ]:
prompt = "Propón un eslogan de máximo 8 palabras para una cafetería de barrio. Responde solo con el eslogan."

for temp in [0.0, 1.0]:
    print(f"--- temperature={temp} ---")
    for _ in range(3):
        r = generar(prompt, temperature=temp, max_output_tokens=40)
        print(" ·", r.text.strip())

💡 **Observa:** con `temperature=0` las tres respuestas se parecen (o son iguales); con 1.0 cambian, aunque alguna puede coincidir con la de temperatura 0: la temperatura alta no garantiza variedad, solo la hace más probable. Para tareas de extracción o clasificación usaremos valores bajos; para creatividad, altos.

## Ejercicios propuestos

De dificultad creciente. El último siempre es abierto. Si te trabas, revisa las pistas y la parte guiada; la solución se publica después de la clase.

### Ejercicio 1 – Tabla comparativa de temperature

Completa la tabla del taller: para un **mismo prompt de tu elección**, genera una respuesta con `temperature` 0.0, 0.7 y 1.0 y guarda los resultados en un DataFrame con columnas `temperature`, `respuesta`, `tokens_salida`.

In [ ]:
import pandas as pd

prompt = "ESCRIBE TU PROMPT AQUÍ"
filas = []
for temp in [0.0, 0.7, 1.0]:
    # TODO: llama a generar(...) con esa temperature y agrega una fila al listado `filas`
    pass

pd.DataFrame(filas)

### Ejercicio 2 – Estimar el costo de 1.000 consultas

Supón que tu asistente recibirá 1.000 consultas al día como la de la Parte guiada 1 y que pasarás al nivel de pago. Estima el costo diario y mensual (30 días) con `MODEL` y compáralo con `gemini-3.8-flash`. ¿Qué pasa con el costo si el system prompt crece a 2.000 tokens?

In [ ]:
# TODO: usa respuesta.usage_metadata y la función costo_usd(...) para calcular
consultas_dia = 1000


### Ejercicio 3 – Pedir una salida en JSON

Pide al modelo que extraiga de una frase libre los campos `producto`, `cantidad` y `ciudad` y los devuelva **solo como JSON**. Gemini permite forzarlo con `response_mime_type="application/json"` en la configuración. Convierte la respuesta con `json.loads` y verifica que sea un diccionario. Prueba con 3 frases distintas, incluida una donde falte un dato.

In [ ]:
import json
frases = ["Necesito 40 cajas de cartón para Valparaíso",
          "Mándame guantes a Antofagasta",
          "ESCRIBE OTRA FRASE"]
# TODO: system_instruction que pida las claves producto, cantidad y ciudad (null si falta), y response_mime_type="application/json"


### Ejercicio 4 – Abierto: una consulta de tu trabajo

Escribe una `system_instruction` para un asistente útil en **tu** área de trabajo y prueba 3 consultas (sin datos confidenciales: recuerda que es el nivel gratuito). Anota qué respondió bien, qué inventó y qué información le faltó. Lo retomaremos en la clase 2.

In [ ]:
instruccion_mia = """ESCRIBE AQUÍ EL ROL Y LAS REGLAS DE TU ASISTENTE"""
consultas = ["...", "...", "..."]
# TODO


## Para seguir en casa

- Tu key gratuita de Gemini sigue funcionando fuera de clase: los límites diarios se reinician cada día (medianoche hora del Pacífico). Puedes ver el uso y los límites de tu proyecto en [AI Studio](https://aistudio.google.com/).
- Si quieres probar otro modelo, cambia `MODEL` (por ejemplo a `"gemini-3.8-flash"`), pero recuerda que los modelos 3.6+ ignoran `temperature`.
- Recuerda la regla del nivel gratuito: nada de datos personales ni confidenciales en los prompts.
- **Lectura relacionada:** Lanham (2024), *AI Agents in Action*, cap. 1; Alto (2025), *AI Agents in Practice*, cap. 1.
